In [19]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Multiple models with caching for feature importance analysis

In [2]:
import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'

# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# At time t (end of month), we only know data for month t-1
fred_md = fred_md_raw.shift(1)  

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Backfill fred_md to avoid nans
fred_md = fred_md.bfill()

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

y_all = monthly_xr[['24','36','48','60','72','84','96','108','120']].values
dates = monthly_xr.index

OOS_start = pd.Timestamp('1990-01-31')
# OOS_start = pd.Timestamp('1972-01-31')

/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from models.base import *
from models.classical import *
from models.other import *
from models.gbt import *
from models.linear import *
from models.tree import *
from models.ann import *
from utils.persistence_utils import run_expanding_multi_seed

def model_factory(seed):
    return ForwardRateANN(archi=(3,), series='forward', 
                              do_grid_search=True, tune_every=60, seed=seed,
                              epochs=100, patience=20)    

seeds = list(range(68, 100))  # 100 seeds

results = run_expanding_multi_seed(
    model_factory=model_factory,
    seeds=seeds,
    X=X,
    y=y_all,                  # can be 1D or multi-output
    dates=dates,
    oos_start=OOS_start,
    run_name="FFNN_3_100seeds_monthly",
    gap=0,
    refit_freq=1,
    benchmark="hist_mean"
)

print("Done. Saved all refit models for all seeds.")

seeds: 100%|██████████| 32/32 [40:31<00:00, 75.98s/it]

Done. Saved all refit models for all seeds.


In [ ]:
import numpy as np
import pandas as pd
from utils.persistence_utils import build_snapshot_index, load_results
import utils.window_utils as wu

results = load_results(
    run_name="FFNN_3_100seeds_monthly",
    seeds=range(100)
)
print(f"Loaded {len(results)} seeds")


def topk_dynamic_ensemble(results, run_name, dates, y_true, k=2, benchmark="hist_mean"):
    idx = build_snapshot_index(run_name=run_name)
    if "val_loss" not in idx.columns:
        raise ValueError("val_loss missing in snapshot metadata; update persistence save callback first.")

    seed_list = sorted(results.keys())
    T = len(dates)

    # Stack forecasts: supports 1D or multi-output
    sample = results[seed_list[0]]["forecast"]
    if sample.ndim == 1:
        fc = np.full((len(seed_list), T), np.nan)
    else:
        fc = np.full((len(seed_list), T, sample.shape[1]), np.nan)

    for i, s in enumerate(seed_list):
        fc[i] = results[s]["forecast"]

    ensemble = np.full_like(sample, np.nan)
    picks = {}

    # Keep only what we need
    loss_df = idx[["seed", "t_index", "val_loss"]].copy()
    loss_df = loss_df.dropna(subset=["val_loss"]).sort_values(["seed", "t_index"])

    for t in range(T):
        # latest available refit loss per seed at time t
        avail = loss_df[loss_df["t_index"] <= t]
        latest = avail.groupby("seed", as_index=False).tail(1)

        top = latest.nsmallest(k, "val_loss")["seed"].tolist()
        if not top:
            continue

        sel_ix = [seed_list.index(s) for s in top]
        if sample.ndim == 1:
            ensemble[t] = np.nanmean(fc[sel_ix, t], axis=0)
        else:
            ensemble[t, :] = np.nanmean(fc[sel_ix, t, :], axis=0)

        picks[t] = top

    r2 = wu.oos_r2(y_true, ensemble, benchmark=benchmark)
    return ensemble, r2, picks

In [12]:
results = rebuild_results_from_snapshots(
    run_name="FFNN_3_100seeds_monthly",
    X=X, y=y_all, dates=dates, oos_start=OOS_start,
    seeds=range(100), gap=0, refit_freq=1, benchmark="hist_mean"
)

100%|██████████| 100/100 [53:12<00:00, 31.92s/it]


In [24]:
import joblib
results = joblib.load("/home/ulrikts/Documents/NTNU/TIO4900-Replication/notebooks/artifacts/models/FFNN_3_100seeds_monthly/results_dict_FFNN_3_100seeds_monthly")

In [25]:
results

{0: {'forecast': array([[        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         ...,
         [ 0.00021576,  0.00188237,  0.0008322 , ..., -0.00010809,
           0.00121634,  0.00142457],
         [ 0.00057522,  0.00078376,  0.00087612, ...,  0.0023886 ,
           0.00287221,  0.00358413],
         [ 0.00110358,  0.00143755,  0.00143198, ...,  0.00230995,
           0.00207213,  0.00196164]], shape=(557, 9)),
  'r2': array([-0.03557111, -0.02751553, -0.0323488 , -0.02614202, -0.01905572,
         -0.01497346, -0.00966645, -0.02461585,  0.00012769])},
 1: {'forecast': array([[        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,

In [16]:
ensemble_fc, ensemble_r2, picks = topk_dynamic_ensemble(
    results=results,
    run_name="FFNN_3_100seeds_monthly",
    dates=dates,
    y_true=y_all,
    k=10,  
    benchmark="hist_mean",
)

print("Top-k ensemble OOS R2:", ensemble_r2)

Top-k ensemble OOS R2: [-0.02505608 -0.01036868 -0.01844416 -0.01146904 -0.01382208 -0.0061738
 -0.01060939 -0.00657127 -0.01295924]
